In [ ]:
# ---------------------------------------------------------------------------
# Path setup — fixes imports and data paths when notebook runs from subdir
# ---------------------------------------------------------------------------
from pathlib import Path
import sys, os

REPO_ROOT = Path().resolve().parent.parent  # notebooks/synthetic/ -> root
os.chdir(REPO_ROOT)                          # all relative paths (cache/, data/, results/, splits/) resolve from root
sys.path.insert(0, str(REPO_ROOT / "src"))   # find rise, crise, synth_gen modules

# Synthetic Probe Generation — Method 2: Stable Diffusion img2img

Generates AI-synthesised versions of real probe images using Stable Diffusion img2img.

**Key distinction from Methods 1 & 3:**  
The source is the *true identity's own probe image* — not a different person's face.  
The generative model recreates the face synthetically, conditioning on the real probe.

**Research question this tests:**  
> When SD generates a new face conditioned on a real probe, does it preserve the identity  
> features ArcFace relies on — or does it produce a plausible-looking face that diverges  
> from the true identity's decision boundary?

**`strength` parameter** (analogous to morphing alpha):
- Low (0.3): output stays close to input — high identity preservation expected
- Mid (0.5): moderate generation drift — primary analysis point
- High (0.7): significant generation — may diverge from ArcFace identity

**Output**: `data/synthetic_probes/sd_img2img/{identity}/{identity}_sdimg2img_{strength_str}_{k}.jpg`  
**Metadata**: appended to `data/synthetic_probes/metadata.csv` (`blend_alpha` = strength value)

In [ ]:
# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

SPLIT_PATH      = "splits/lfw_1N_split.json"
GALLERY_EMB     = "cache/G.npy"
GALLERY_IDS     = "cache/gallery_ids.npy"
CRISE_SUMMARY   = "results/crise_maps/summary_crise_tau0.1_N1000_s8_p0.5_MASTERSEED123.csv"

SD_MODEL_ID     = "runwayml/stable-diffusion-v1-5"
PROMPT          = "a realistic portrait photo of a person, photographic, high quality"
NEG_PROMPT      = "cartoon, anime, painting, blurry, low quality, artifacts, deformed"
GUIDANCE_SCALE  = 7.5
INFERENCE_STEPS = 30

OUT_DIR         = "data/synthetic_probes/sd_img2img"
META_PATH       = "data/synthetic_probes/metadata.csv"
GEN_METHOD      = "sd_img2img"

STRENGTHS       = [0.3, 0.5, 0.7]   # generation drift; primary = 0.5
GEN_SEED        = 42

SYNTH_SEED      = 42
N_TARGET_IDS    = 50
N_PROBES_PER_ID = 3

In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------

import os, json
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image
import torch
from diffusers import StableDiffusionImg2ImgPipeline
from insightface.app import FaceAnalysis

from synth_gen import (
    get_embedding_from_image,
    select_target_identities,
    select_probes_for_identity,
)

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(META_PATH), exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

In [ ]:
# ---------------------------------------------------------------------------
# Load SD img2img pipeline
# First run downloads ~4GB; subsequent runs load from cache.
# ---------------------------------------------------------------------------

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    SD_MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    safety_checker=None,          # disable NSFW filter for faces
    requires_safety_checker=False,
).to(DEVICE)
pipe.set_progress_bar_config(disable=True)
print(f"SD pipeline loaded: {SD_MODEL_ID}")

In [ ]:
# ---------------------------------------------------------------------------
# InsightFace setup (for embedding synthetic probes)
# ---------------------------------------------------------------------------

app = FaceAnalysis(
    name="buffalo_l",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
)
app.prepare(ctx_id=0, det_size=(640, 640))
rec = app.models["recognition"]
print("InsightFace ready")

In [ ]:
# ---------------------------------------------------------------------------
# Load split + gallery
# ---------------------------------------------------------------------------

with open(SPLIT_PATH) as f:
    split = json.load(f)

G           = np.load(GALLERY_EMB).astype(np.float32)
gallery_ids = np.load(GALLERY_IDS, allow_pickle=True).tolist()
id_to_index = {gid: i for i, gid in enumerate(gallery_ids)}

print(f"Gallery: {G.shape[0]} identities")

In [ ]:
# ---------------------------------------------------------------------------
# Select 50 target identities  (same seed → same set as Methods 1 & 3)
# ---------------------------------------------------------------------------

rise_summary = pd.read_csv(CRISE_SUMMARY)
id_col = "true_id" if "true_id" in rise_summary.columns else rise_summary.columns[0]
completed_ids = rise_summary[id_col].dropna().unique().tolist()

target_ids = select_target_identities(completed_ids, n=N_TARGET_IDS, seed=SYNTH_SEED)
print(f"Target identities: {len(target_ids)} (seed={SYNTH_SEED})")
print(f"Strengths: {STRENGTHS}  →  {len(target_ids) * N_PROBES_PER_ID * len(STRENGTHS)} total probes")

In [ ]:
# ---------------------------------------------------------------------------
# Generation helper
# ---------------------------------------------------------------------------

def generate_sd_img2img(probe_path: str, strength: float, seed: int) -> Image.Image | None:
    """
    Run SD img2img on a single probe image.
    Resizes to 512×512 for SD, returns PIL Image at original size.
    Returns None on failure.
    """
    try:
        img = Image.open(probe_path).convert("RGB")
        orig_size = img.size
        img_512 = img.resize((512, 512), Image.LANCZOS)

        generator = torch.Generator(DEVICE).manual_seed(seed)
        result = pipe(
            prompt=PROMPT,
            negative_prompt=NEG_PROMPT,
            image=img_512,
            strength=strength,
            guidance_scale=GUIDANCE_SCALE,
            num_inference_steps=INFERENCE_STEPS,
            generator=generator,
        ).images[0]

        return result.resize(orig_size, Image.LANCZOS)
    except Exception as e:
        print(f"  [SD error] {e}")
        return None

In [ ]:
# ---------------------------------------------------------------------------
# Main generation loop — all strengths in one pass
# ---------------------------------------------------------------------------

records = []
n_ok = 0
n_fail = 0

for exp_i, target_id in enumerate(tqdm(target_ids, desc="Target identities")):
    id_out_dir = os.path.join(OUT_DIR, target_id)
    os.makedirs(id_out_dir, exist_ok=True)

    # Pick N probe images from this identity's own probe set
    probe_paths = select_probes_for_identity(
        target_id, exp_i, split, n=N_PROBES_PER_ID, seed=SYNTH_SEED
    )

    for k, probe_path in enumerate(probe_paths):
        for strength in STRENGTHS:
            strength_str = str(strength).replace(".", "p")  # e.g. "0p5"
            # Per-sample seed: varies by identity, probe index, and strength
            sample_seed = GEN_SEED * 10_000 + exp_i * 100 + k * 10 + int(strength * 10)

            result_pil = generate_sd_img2img(probe_path, strength, sample_seed)

            if result_pil is None:
                n_fail += 1
                records.append(dict(
                    identity=target_id, generation_method=GEN_METHOD,
                    source_identity=target_id, source_path=probe_path,
                    blend_alpha=strength, output_path=None, embedding_ok=False,
                    arcface_similarity=None, rank1_match=None,
                    saliency_cosine_sim=None, saliency_l1=None, case_label=None,
                ))
                continue

            out_fname = f"{target_id}_sdimg2img_{strength_str}_{k}.jpg"
            out_path  = os.path.join(id_out_dir, out_fname)
            result_pil.save(out_path, quality=95)

            n_ok += 1
            records.append(dict(
                identity=target_id, generation_method=GEN_METHOD,
                source_identity=target_id, source_path=probe_path,
                blend_alpha=strength, output_path=out_path, embedding_ok=True,
                arcface_similarity=None, rank1_match=None,
                saliency_cosine_sim=None, saliency_l1=None, case_label=None,
            ))

print(f"\nGeneration complete: {n_ok} saved, {n_fail} failed")

In [ ]:
# ---------------------------------------------------------------------------
# Visualisation: real probe vs generated at each strength
# Rows = first 4 identities;  Cols = real | s=0.3 | s=0.5 | s=0.7
# ---------------------------------------------------------------------------

ok_records  = [r for r in records if r["output_path"] is not None]
show_targets = target_ids[:4]

fig, axes = plt.subplots(len(show_targets), 4,
                         figsize=(11, 3 * len(show_targets)))

for row, tid in enumerate(show_targets):
    tid_recs = [r for r in ok_records if r["identity"] == tid]
    ref_rec  = next((r for r in tid_recs if r["blend_alpha"] == 0.5), None)
    if ref_rec is None:
        for col in range(4): axes[row, col].axis("off")
        continue

    real_img = cv2.cvtColor(cv2.imread(ref_rec["source_path"]), cv2.COLOR_BGR2RGB)

    gen_imgs = {}
    for s in STRENGTHS:
        r = next((r for r in tid_recs
                  if r["blend_alpha"] == s
                  and r["source_path"] == ref_rec["source_path"]
                  and r["output_path"] is not None), None)
        if r:
            gen_imgs[s] = cv2.cvtColor(cv2.imread(r["output_path"]), cv2.COLOR_BGR2RGB)

    panels = [
        (real_img,           f"Real probe\n{tid[:20]}"),
        (gen_imgs.get(0.3),  "SD strength=0.3"),
        (gen_imgs.get(0.5),  "SD strength=0.5"),
        (gen_imgs.get(0.7),  "SD strength=0.7"),
    ]
    for col, (img, title) in enumerate(panels):
        if img is not None:
            axes[row, col].imshow(img)
        axes[row, col].set_title(title, fontsize=8)
        axes[row, col].axis("off")

plt.suptitle("SD img2img — Real | strength=0.3 | 0.5 | 0.7", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "sample_grid.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Compute ArcFace embeddings
# ---------------------------------------------------------------------------

synth_embeddings = {}

for r in tqdm(ok_records, desc="Embedding generated probes"):
    emb = get_embedding_from_image(r["output_path"], app, rec)
    synth_embeddings[r["output_path"]] = emb
    if emb is None:
        r["embedding_ok"] = False

n_emb_ok   = sum(1 for e in synth_embeddings.values() if e is not None)
n_emb_fail = sum(1 for e in synth_embeddings.values() if e is None)
print(f"Embeddings: {n_emb_ok} OK, {n_emb_fail} failed (no face detected in generated image)")

In [ ]:
# ---------------------------------------------------------------------------
# Compute rank-1 match + cosine similarity — per strength
# ---------------------------------------------------------------------------

for r in records:
    if not r["embedding_ok"] or r["output_path"] is None:
        continue
    emb = synth_embeddings.get(r["output_path"])
    if emb is None:
        r["embedding_ok"] = False
        continue

    sims     = G @ emb
    true_idx = id_to_index[r["identity"]]
    r["arcface_similarity"] = round(float(sims[true_idx]), 6)
    r["rank1_match"]        = (int(np.argmax(sims)) == true_idx)

print("=== Rank-1 rate and mean similarity per strength ===")
for s in STRENGTHS:
    rows_s = [r for r in records if r["blend_alpha"] == s and r["rank1_match"] is not None]
    if not rows_s:
        continue
    r1 = sum(r["rank1_match"] for r in rows_s) / len(rows_s)
    ms = np.mean([r["arcface_similarity"] for r in rows_s])
    print(f"  strength={s}:  rank-1={r1:.4f}  mean_sim={ms:.4f}  n={len(rows_s)}")

In [ ]:
# ---------------------------------------------------------------------------
# Append to shared metadata.csv
# ---------------------------------------------------------------------------

META_COLS = [
    "identity", "generation_method", "source_identity", "blend_alpha",
    "output_path", "embedding_ok", "arcface_similarity", "rank1_match",
    "saliency_cosine_sim", "saliency_l1", "case_label",
]

new_df = pd.DataFrame(records)[META_COLS]

if os.path.exists(META_PATH):
    existing = pd.read_csv(META_PATH)
    # Drop stale sd_img2img rows and also any leftover simswap rows
    existing = existing[~existing["generation_method"].isin([GEN_METHOD, "simswap"])]
    combined = pd.concat([existing, new_df], ignore_index=True)
else:
    combined = new_df

combined.to_csv(META_PATH, index=False)
print(f"metadata.csv: {len(combined)} total rows → {META_PATH}")
print(combined.groupby("generation_method")[["embedding_ok", "rank1_match"]].mean().round(3))

In [ ]:
# ---------------------------------------------------------------------------
# Strength ablation plot
# ---------------------------------------------------------------------------

strength_stats = []
for s in STRENGTHS:
    rows_s = [r for r in records if r["blend_alpha"] == s and r["rank1_match"] is not None]
    if rows_s:
        strength_stats.append(dict(
            strength=s,
            rank1_rate=sum(r["rank1_match"] for r in rows_s) / len(rows_s),
            mean_sim=np.mean([r["arcface_similarity"] for r in rows_s]),
            std_sim=np.std([r["arcface_similarity"] for r in rows_s]),
        ))

if strength_stats:
    df_s = pd.DataFrame(strength_stats)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.5))

    ax1.plot(df_s["strength"], df_s["rank1_rate"], "o-", color="tab:blue")
    ax1.set_xlabel("SD strength")
    ax1.set_ylabel("Rank-1 match rate")
    ax1.set_title("Identity preservation vs generation strength")
    ax1.set_xticks(STRENGTHS)
    ax1.grid(True, alpha=0.3)

    ax2.errorbar(df_s["strength"], df_s["mean_sim"], yerr=df_s["std_sim"],
                 fmt="o-", color="tab:orange", capsize=4)
    ax2.set_xlabel("SD strength")
    ax2.set_ylabel("Mean cosine sim to true identity")
    ax2.set_title("ArcFace similarity vs generation strength")
    ax2.set_xticks(STRENGTHS)
    ax2.grid(True, alpha=0.3)

    plt.suptitle("SD img2img — Strength Ablation", fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "strength_ablation.png"), dpi=150, bbox_inches="tight")
    plt.show()